# Setup: imports and config hash

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")

In [2]:
import sys
import os
import glob
import warnings
import numpy as np

warnings.filterwarnings(
    "ignore",
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated",
    category=FutureWarning,
)

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
    print("kaggle credentials loaded from secrets")
except Exception as exc:
    print("kaggle secrets not loaded:", exc)

try:
    import core
except ModuleNotFoundError:
    hits = glob.glob("/kaggle/input/**/core.py", recursive=True)
    if not hits:
        raise ModuleNotFoundError("core.py not found.")
    sys.path.insert(0, os.path.dirname(hits[0]))
    import core

print("import-time CONFIG_HASH:", core.CONFIG_HASH)

kaggle credentials loaded from secrets
import-time CONFIG_HASH: 03fa342b535795c1


# Environment

In [3]:
env = core.log_environment()

[10:51:31] env python: 3.13.15
[10:51:31] env torch: 2.11.0+cu128
[10:51:31] env torchvision: 0.26.0+cu128
[10:51:31] env numpy: 2.1.3
[10:51:31] env sklearn: 1.6.1
[10:51:31] env pandas: 2.3.3
[10:51:31] env cuda: 12.8
[10:51:31] env gpu: Tesla T4
[10:51:31] env os: Linux-6.18.48+-x86_64-with-glibc2.39
[10:51:31] env git_commit: unavailable
[10:51:31] env git_state: unavailable
[10:51:31] env config_hash: 03fa342b535795c1


# Load data, verify counts, confirm balanced eval splits

In [4]:
data = core.load_pcam()
c = data.counts

print("raw counts")
print(f"  train pos/neg: {c['train_pos']} / {c['train_neg']}")
print(f"  valid pos/neg: {c['valid_pos_raw']} / {c['valid_neg_raw']}")
print(f"  test  pos/neg: {c['test_pos_raw']} / {c['test_neg_raw']}")

assert (c["train_pos"], c["train_neg"]) == (131072, 131072), "train raw counts mismatch"
assert (c["valid_pos_raw"], c["valid_neg_raw"]) == (16369, 16399), "valid raw counts mismatch"
assert (c["test_pos_raw"], c["test_neg_raw"]) == (16377, 16391), "test raw counts mismatch"

valid_pos = int((data.valid_labels == 1).sum())
valid_neg = int((data.valid_labels == 0).sum())
test_pos = int((data.test_labels == 1).sum())
test_neg = int((data.test_labels == 0).sum())

print("balanced counts")
print(f"  valid pos/neg: {valid_pos} / {valid_neg} (total {len(data.valid_labels)})")
print(f"  test  pos/neg: {test_pos} / {test_neg} (total {len(data.test_labels)})")

assert (valid_pos, valid_neg) == (16369, 16369), "valid not balanced to 16,369/16,369"
assert (test_pos, test_neg) == (16377, 16377), "test not balanced to 16,377/16,377"
assert len(data.valid_labels) == 32738, "valid balanced size != 32,738"
assert len(data.test_labels) == 32754, "test balanced size != 32,754"

import pandas as pd
vk = os.path.join(core.PATHS["balanced_eval"], "valid_kept.csv")
tk = os.path.join(core.PATHS["balanced_eval"], "test_kept.csv")
assert os.path.exists(vk) and os.path.exists(tk), "balanced_eval kept CSVs missing"
n_vk = len(pd.read_csv(vk))
n_tk = len(pd.read_csv(tk))
assert n_vk == 32738, f"valid_kept.csv length {n_vk} != 32,738"
assert n_tk == 32754, f"test_kept.csv length {n_tk} != 32,754"
print(f"kept index files present: valid_kept={n_vk}, test_kept={n_tk}")

[10:54:21] train pos/neg 131072/131072
[10:54:21] valid pos/neg (raw) 16369/16399
[10:54:21] test pos/neg (raw) 16377/16391
[10:54:21] valid balanced 32738 (16369/16369)
[10:54:21] test balanced 32754 (16377/16377)
raw counts
  train pos/neg: 131072 / 131072
  valid pos/neg: 16369 / 16399
  test  pos/neg: 16377 / 16391
balanced counts
  valid pos/neg: 16369 / 16369 (total 32738)
  test  pos/neg: 16377 / 16377 (total 32754)
kept index files present: valid_kept=32738, test_kept=32754


# Norm-stat provenance gate

In [5]:
mean, std = core.compute_norm_stats(data.train_images)

ref_mean = core.NORM_MEAN
ref_std = core.NORM_STD
TOL = 1e-3
channels = ("R", "G", "B")

print("recomputed vs reference (abs tolerance 1e-3)")
offending = []
for i, ch in enumerate(channels):
    dm = abs(mean[i] - ref_mean[i])
    ds = abs(std[i] - ref_std[i])
    print(f"  {ch} mean: {mean[i]:.6f} vs {ref_mean[i]:.6f}  (|diff|={dm:.2e})")
    print(f"  {ch} std : {std[i]:.6f} vs {ref_std[i]:.6f}  (|diff|={ds:.2e})")
    if dm > TOL:
        offending.append(f"{ch} mean (|diff|={dm:.2e})")
    if ds > TOL:
        offending.append(f"{ch} std (|diff|={ds:.2e})")

if offending:
    raise ValueError("pixel-provenance gate FAILED: " + "; ".join(offending))

print("provenance gate PASSED; recomputed values are authoritative")

recomputed vs reference (abs tolerance 1e-3)
  R mean: 0.700756 vs 0.700756  (|diff|=1.73e-09)
  R std : 0.234978 vs 0.234978  (|diff|=1.18e-07)
  G mean: 0.538358 vs 0.538358  (|diff|=1.96e-07)
  G std : 0.277409 vs 0.277409  (|diff|=1.15e-07)
  B mean: 0.691621 vs 0.691621  (|diff|=4.44e-07)
  B std : 0.212895 vs 0.212895  (|diff|=4.09e-07)
provenance gate PASSED; recomputed values are authoritative


# Freeze config, then refresh (mandatory sequence)

In [6]:
path = core.write_frozen_config(mean, std)
print("wrote", path)

h = core.refresh_config()
print("active CONFIG_HASH after refresh:", h)
print("core.NORM_MEAN now:", core.NORM_MEAN)
print("core.NORM_STD  now:", core.NORM_STD)

file_hash = None
with open(path, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith("- config_hash:"):
            file_hash = line.split(":", 1)[1].strip()
            break

assert file_hash is not None, "config_hash not found in frozen_config.md"
assert file_hash == core.CONFIG_HASH, (
    f"hash mismatch: file {file_hash} vs active {core.CONFIG_HASH}")
print("frozen_config.md hash matches active hash:", file_hash)

wrote /kaggle/working/frozen_config.md
active CONFIG_HASH after refresh: 03fa342b535795c1
core.NORM_MEAN now: [0.700756, 0.538358, 0.691621]
core.NORM_STD  now: [0.234978, 0.277409, 0.212895]
frozen_config.md hash matches active hash: 03fa342b535795c1


# Shared-condition unit test: (32,000, 32,000) across A, B, D

In [7]:
a_min, a_maj = 32000, 32000
b_min = int(round(core.N_F / (1 + 1)))
b_maj = core.N_F - b_min
d_min, d_maj = 32000, 32000

assert (a_min, a_maj) == (32000, 32000), "A counts drift"
assert (b_min, b_maj) == (32000, 32000), f"B counts drift: {(b_min, b_maj)}"
assert (d_min, d_maj) == (32000, 32000), "D counts drift"

cid = {
    "A": core.make_condition_id(a_min, a_maj),
    "B": core.make_condition_id(b_min, b_maj),
    "D": core.make_condition_id(d_min, d_maj),
}
rid = {
    "A": core.make_run_id("cnn", a_min, a_maj, 21, 21, 21),
    "B": core.make_run_id("cnn", b_min, b_maj, 21, 21, 21),
    "D": core.make_run_id("cnn", d_min, d_maj, 21, 21, 21),
}
assert cid["A"] == cid["B"] == cid["D"], f"condition_id differs: {cid}"
assert rid["A"] == rid["B"] == rid["D"], f"run_id differs: {rid}"

sub_a = core.make_subset(a_min, a_maj, 21, data)
sub_b = core.make_subset(b_min, b_maj, 21, data)
sub_d = core.make_subset(d_min, d_maj, 21, data)
assert np.array_equal(sub_a, sub_b), "A vs B subset differs"
assert np.array_equal(sub_a, sub_d), "A vs D subset differs"

print("shared-condition unit test PASS")
print("  condition_id:", cid["A"])
print("  run_id:", rid["A"])
print("  subset length:", len(sub_a))

shared-condition unit test PASS
  condition_id: m32000_M32000
  run_id: cnn__m32000_M32000__d21_i21_l21
  subset length: 64000


# Setup summary

In [8]:
core._ensure(core.PATHS["summaries"])
summary_path = os.path.join(core.PATHS["summaries"], "00_setup.md")

lines = [
    "# Notebook 00: setup summary",
    "",
    f"Active config hash: {core.CONFIG_HASH}",
    "",
    "## Raw class counts",
    f"- train pos/neg: {c['train_pos']} / {c['train_neg']}",
    f"- valid pos/neg: {c['valid_pos_raw']} / {c['valid_neg_raw']}",
    f"- test pos/neg: {c['test_pos_raw']} / {c['test_neg_raw']}",
    "",
    "## Balanced evaluation splits",
    f"- valid: {valid_pos} / {valid_neg} (total {len(data.valid_labels)})",
    f"- test: {test_pos} / {test_neg} (total {len(data.test_labels)})",
    f"- kept index files: valid_kept={n_vk}, test_kept={n_tk}",
    "",
    "## Normalisation provenance (tolerance 1e-3, all channels within)",
    f"- mean recomputed: {mean[0]:.6f}, {mean[1]:.6f}, {mean[2]:.6f}",
    f"- mean reference: {ref_mean[0]:.6f}, {ref_mean[1]:.6f}, {ref_mean[2]:.6f}",
    f"- std recomputed: {std[0]:.6f}, {std[1]:.6f}, {std[2]:.6f}",
    f"- std reference: {ref_std[0]:.6f}, {ref_std[1]:.6f}, {ref_std[2]:.6f}",
    "- gate result: PASSED",
    "",
    "## Shared-condition unit test",
    f"- condition_id: {cid['A']}",
    f"- run_id (cnn, seeds 21/21/21): {rid['A']}",
    f"- subset length: {len(sub_a)}",
    "- A/B/D identical: PASS",
    "",
]
with open(summary_path, "w") as f:
    f.write("\n".join(lines))

print("wrote", summary_path)

wrote /kaggle/working/summaries/00_setup.md


# Persistence

In [9]:
if core.DATAWISE_RESULTS_SLUG is not None:
    core.push_results_store()
    print("pushed results store to", core.DATAWISE_RESULTS_SLUG)
else:
    print("DATAWISE_RESULTS_SLUG is unset; persistence skipped this session")

Starting upload for file __notebook__.ipynb


100%|██████████| 21.0k/21.0k [00:00<00:00, 44.7kB/s]


Upload successful: __notebook__.ipynb (21KB)
Starting upload for file logs.zip


100%|██████████| 583/583 [00:00<00:00, 1.27kB/s]


Upload successful: logs.zip (583B)
Starting upload for file balanced_eval.zip


...

Upload successful: balanced_eval.zip (139KB)
Starting upload for file frozen_config.md


100%|██████████| 431/431 [00:00<00:00, 994B/s]


Upload successful: frozen_config.md (431B)
Starting upload for file summaries.zip


100%|██████████| 547/547 [00:00<00:00, 1.29kB/s]


Upload successful: summaries.zip (547B)
pushed results store to hrushisanap/datawise-results
